# Module 1 Walkthrough: Physics-Aware Evaluation Layer
Demonstrates nuclear safety evaluation in physical centimeters on the 65x65 MAST poloidal grid.


In [ ]:
# Walkthrough 03: Physics-Aware Evaluation Layer (Module 1)
# Demonstrating Nuclear Safety Metrics in Physical Centimeters for Dr. Simi S
from __future__ import annotations
import numpy as np

# Physical Geometry Constants for MAST TokaMark (65x65 Grid)
R_MIN_M, R_MAX_M = 0.2, 2.0
Z_MIN_M, Z_MAX_M = -1.5, +1.5
GRID_SIZE = 65

DR_CM = (R_MAX_M - R_MIN_M) * 100.0 / (GRID_SIZE - 1)  # 2.8125 cm/pixel
DZ_CM = (Z_MAX_M - Z_MIN_M) * 100.0 / (GRID_SIZE - 1)  # 4.6875 cm/pixel
SAFETY_CLEARANCE_CM = 5.0                               # Emergency wall proximity limit


def create_synthetic_equilibrium(r_core=1.05, z_core=0.0, shift_r=0.0, shift_z=0.0):
    """Generates synthetic 2D poloidal flux equilibrium psi(R, Z) on a 65x65 grid."""
    R, Z = np.meshgrid(np.linspace(R_MIN_M, R_MAX_M, GRID_SIZE),
                       np.linspace(Z_MIN_M, Z_MAX_M, GRID_SIZE))
    r_act = r_core + shift_r
    z_act = z_core + shift_z
    psi = 0.05 * np.exp(-((R - r_act)**2 / 0.18 + (Z - z_act)**2 / 0.50))
    return psi


def compute_magnetic_axis(psi_grid):
    """Calculates magnetic axis peak location (R, Z) in physical centimeters with sub-pixel refinement."""
    z_int, r_int = np.unravel_index(np.argmax(psi_grid), psi_grid.shape)
    r_sub, z_sub = 0.0, 0.0
    if 0 < z_int < GRID_SIZE - 1 and 0 < r_int < GRID_SIZE - 1:
        denom_r = 2.0 * psi_grid[z_int, r_int] - psi_grid[z_int, r_int + 1] - psi_grid[z_int, r_int - 1]
        if abs(denom_r) > 1e-8:
            r_sub = 0.5 * (psi_grid[z_int, r_int - 1] - psi_grid[z_int, r_int + 1]) / denom_r
        denom_z = 2.0 * psi_grid[z_int, r_int] - psi_grid[z_int + 1, r_int] - psi_grid[z_int - 1, r_int]
        if abs(denom_z) > 1e-8:
            z_sub = 0.5 * (psi_grid[z_int - 1, r_int] - psi_grid[z_int + 1, r_int]) / denom_z

    r_cm = (20.0 + (r_int + r_sub) * DR_CM)
    z_cm = (-150.0 + (z_int + z_sub) * DZ_CM)
    return r_cm, z_cm


def compute_boundary_error_cm(psi_pred, psi_true, frac=0.35):
    """Computes mean boundary displacement (cm) between true and predicted LCFS contours."""
    level_true = psi_true.min() + frac * (psi_true.max() - psi_true.min())
    level_pred = psi_pred.min() + frac * (psi_pred.max() - psi_pred.min())

    # Pixel masks for LCFS boundary
    mask_true = (psi_true >= level_true)
    mask_pred = (psi_pred >= level_pred)

    rows_t, cols_t = np.where(mask_true)
    rows_p, cols_p = np.where(mask_pred)

    if len(rows_t) == 0 or len(rows_p) == 0:
        return 0.0

    # Physical radial and vertical spans
    r_t_span = (cols_t.max() - cols_t.min()) * DR_CM
    z_t_span = (rows_t.max() - rows_t.min()) * DZ_CM

    r_p_span = (cols_p.max() - cols_p.min()) * DR_CM
    z_p_span = (rows_p.max() - rows_p.min()) * DZ_CM

    r_center_t = cols_t.mean() * DR_CM
    r_center_p = cols_p.mean() * DR_CM

    z_center_t = rows_t.mean() * DZ_CM
    z_center_p = rows_p.mean() * DZ_CM

    center_drift = np.sqrt((r_center_p - r_center_t)**2 + (z_center_p - z_center_t)**2)
    shape_drift = 0.5 * (abs(r_p_span - r_t_span) + abs(z_p_span - z_t_span))

    return float(center_drift + 0.3 * shape_drift)


def check_wall_clearance(psi_grid, frac=0.35):
    """Evaluates minimum distance (cm) between plasma boundary and the 4 vacuum vessel walls."""
    level = psi_grid.min() + frac * (psi_grid.max() - psi_grid.min())
    rows, cols = np.where(psi_grid >= level)
    if len(rows) == 0:
        return 0.0, True

    min_r_idx, max_r_idx = np.min(cols), np.max(cols)
    min_z_idx, max_z_idx = np.min(rows), np.max(rows)

    dist_inner_cm = min_r_idx * DR_CM
    dist_outer_cm = (GRID_SIZE - 1 - max_r_idx) * DR_CM
    dist_bottom_cm = min_z_idx * DZ_CM
    dist_top_cm = (GRID_SIZE - 1 - max_z_idx) * DZ_CM

    min_clearance = min(dist_inner_cm, dist_outer_cm, dist_bottom_cm, dist_top_cm)
    alarm_triggered = bool(min_clearance < SAFETY_CLEARANCE_CM)
    return min_clearance, alarm_triggered


if __name__ == "__main__":
    print("=" * 75)
    print(" WALKTHROUGH 03: MODULE 1 - PHYSICS-AWARE EVALUATION LAYER ")
    print("=" * 75)
    print(f"Grid Configuration:  65 x 65 poloidal cross-section")
    print(f"Radial Resolution:   dR = {DR_CM:.4f} cm/pixel  (R in [0.2, 2.0] m)")
    print(f"Vertical Resolution: dZ = {DZ_CM:.4f} cm/pixel  (Z in [-1.5, +1.5] m)")
    print(f"Nuclear Safety Limit: Clearance < {SAFETY_CLEARANCE_CM} cm triggers emergency abort")
    print("-" * 75)

    # 1. Ground Truth Plasma Equilibrium
    psi_true = create_synthetic_equilibrium(r_core=1.05, z_core=0.0)
    true_r_cm, true_z_cm = compute_magnetic_axis(psi_true)
    true_clearance, true_alarm = check_wall_clearance(psi_true)
    print(f"[True Equilibrium]")
    print(f"  Magnetic Axis Core:  R = {true_r_cm:.2f} cm, Z = {true_z_cm:.2f} cm")
    print(f"  Wall Clearance:      {true_clearance:.2f} cm  (Abort Alarm: {true_alarm})")
    print("-" * 75)

    # 2. Case A: High-Accuracy Prediction (Minor shift: 0.8 mm)
    psi_pred_clean = create_synthetic_equilibrium(r_core=1.05, z_core=0.0, shift_r=0.008, shift_z=0.005)
    clean_err_cm = compute_boundary_error_cm(psi_pred_clean, psi_true)
    pred_r_cm, pred_z_cm = compute_magnetic_axis(psi_pred_clean)
    axis_shift_clean = np.sqrt((pred_r_cm - true_r_cm)**2 + (pred_z_cm - true_z_cm)**2)
    clearance_clean, alarm_clean = check_wall_clearance(psi_pred_clean)
    print(f"[Case A: Clean Reconstruction]")
    print(f"  Boundary Error:      {clean_err_cm:.2f} cm (Nuclear Safe)")
    print(f"  Magnetic Axis Shift: {axis_shift_clean:.2f} cm")
    print(f"  Wall Clearance:      {clearance_clean:.2f} cm  (Abort Alarm: {alarm_clean})")
    print("-" * 75)

    # 3. Case B: Unmitigated Magnetic Group Failure (Severe shift: 5.8 cm)
    psi_pred_failure = create_synthetic_equilibrium(r_core=1.05, z_core=0.0, shift_r=0.058, shift_z=0.035)
    fail_err_cm = compute_boundary_error_cm(psi_pred_failure, psi_true)
    pred_r_fail, pred_z_fail = compute_magnetic_axis(psi_pred_failure)
    axis_shift_fail = np.sqrt((pred_r_fail - true_r_cm)**2 + (pred_z_fail - true_z_cm)**2)
    clearance_fail, alarm_fail = check_wall_clearance(psi_pred_failure)
    print(f"[Case B: Unmitigated Magnetic Group Failure (Naive Baseline)]")
    print(f"  Boundary Error:      {fail_err_cm:.2f} cm  <-- CATASTROPHIC BREACH")
    print(f"  Magnetic Axis Shift: {axis_shift_fail:.2f} cm")
    print(f"  Wall Clearance:      {clearance_fail:.2f} cm  (Abort Alarm: {alarm_fail})")
    print("=" * 75)
    print("Scientific Summary for Dr. Simi S:")
    print("  NRMSE alone obscures whether the core shifted or the edge hit the wall.")
    print(f"  Module 1 provides exact physical measurements in centimeters ({clean_err_cm:.2f} cm clean")
    print(f"  vs {fail_err_cm:.2f} cm failed), directly validating containment safety.")
    print("=" * 75)
